In [2]:
!pip uninstall -y langchain langchain-core langgraph langgraph-prebuilt


Found existing installation: langchain 0.3.27
Uninstalling langchain-0.3.27:
  Successfully uninstalled langchain-0.3.27
Found existing installation: langchain-core 0.3.81
Uninstalling langchain-core-0.3.81:
  Successfully uninstalled langchain-core-0.3.81


In [3]:
!pip install -q \
    unsloth \
    sentence-transformers \
    faiss-cpu \
    langchain \
    langchain-community \
    pypdf \
    python-docx \
    requests==2.32.4 \
    torch==2.9.0 \
    torchaudio==2.9.0


In [5]:
from google.colab import userdata
import os

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")


Upload Documents

In [6]:
from google.colab import files

print("""
📄 Upload Instructions:
• Upload ONLY PDF or Word (.docx) files
• Multiple files allowed
""")

uploaded_files = files.upload()



📄 Upload Instructions:
• Upload ONLY PDF or Word (.docx) files
• Multiple files allowed



Saving Family Medical Leave Act.pdf to Family Medical Leave Act.pdf


Load PDF / Word Files

In [7]:
from langchain.document_loaders import PyPDFLoader, Docx2txtLoader

documents = []

for file in uploaded_files:
    if file.lower().endswith(".pdf"):
        documents.extend(PyPDFLoader(file).load())
    elif file.lower().endswith(".docx"):
        documents.extend(Docx2txtLoader(file).load())
    else:
        print(f"❌ Unsupported file skipped: {file}")

print(f"✅ Loaded {len(documents)} pages")


✅ Loaded 1 pages


Chunk Text

In [8]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

chunks = splitter.split_documents(documents)
print(f"✅ Created {len(chunks)} chunks")


✅ Created 8 chunks


Embeddings (bge-base-en)

In [9]:
from langchain.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True}
)


/tmp/ipython-input-2468189344.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

FAISS Vector Store

In [10]:
from langchain.vectorstores import FAISS

vectorstore = FAISS.from_documents(chunks, embedding_model)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("✅ FAISS vector database ready")


✅ FAISS vector database ready


Load Unsloth LLaMA-3.1-8B Alpaca (4-bit)

In [12]:
from unsloth import FastLanguageModel
import torch

model_name = "unsloth/Meta-Llama-3.1-8B"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = 4096,
    load_in_4bit = True,
    token = os.environ["HF_TOKEN"],
)

FastLanguageModel.for_inference(model)

print("✅ Unsloth LLaMA-3.1-8B Alpaca loaded")


==((====))==  Unsloth 2025.12.9: Fast Llama patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
✅ Unsloth LLaMA-3.1-8B Alpaca loaded


RAG Prompt Template (Alpaca Style)

In [13]:
def build_prompt(context, question):
    return f"""### Instruction:
Answer the question using ONLY the context below.
If the answer is not present, say "I don’t know."

### Context:
{context}

### Question:
{question}

### Response:
"""


Chatbot Function

In [14]:
def chatbot(question):
    docs = retriever.get_relevant_documents(question)
    context = "\n\n".join([doc.page_content for doc in docs])

    prompt = build_prompt(context, question)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.2,
            top_p=0.9,
            do_sample=True
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response.split("### Response:")[-1].strip()


Interactive Chat

In [15]:
while True:
    query = input("\n❓ Ask a question (type 'exit' to stop): ")
    if query.lower() == "exit":
        break
    print("\n🤖 Answer:\n", chatbot(query))



❓ Ask a question (type 'exit' to stop): How Many  FMLA leave  can be taken in a year


/tmp/ipython-input-1495689719.py:2: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  docs = retriever.get_relevant_documents(question)



🤖 Answer:
 12 workweeks of FMLA leave in a 12-month period for:
   • The birth, adoption or foster placement of a child with you,
   • Your serious mental or physical health condition that makes you 
unable to work,
   • To care for your spouse, child or parent with a serious mental or 
physical health condition, and
   • Certain qualifying reasons related to the foreign deployment of your 
spouse, child or parent who is a military servicemember. 
An eligible employee who is the spouse, child, parent or next of kin of a 
covered servicemember with a serious injury or illness may take up to 
26 workweeks of FMLA leave in a single 12-month period to care for the 
servicemember. 
You have the right to use FMLA leave in one block of time. When it is

### Instruction:
Answer the question using ONLY the context below.
If the answer is not present, say "I don’t know."

### Context:
servicemember. 
You have the right to use FMLA leave in one block of time. When it is 
medically necessary or o